In [3]:
import pandas as pd
# Load STRING file
string_df = pd.read_csv("9606.protein.links.v12.0.txt", sep=" ")
string_df.head()
string_df["protein1"] = string_df["protein1"].str.replace("9606.", "", regex=False)
string_df["protein2"] = string_df["protein2"].str.replace("9606.", "", regex=False)



In [4]:
string_df = string_df[string_df["combined_score"] >= 700]
print("Total high confidence interactions:", len(string_df))
mapping_df = pd.read_csv("idmapping_2026_02_16.tsv", sep="\t")
# Rename mapping columns
mapping_df = mapping_df.rename(columns={"From": "protein", "Entry": "uniprot"})

# Merge for protein1
string_df = string_df.merge(mapping_df, left_on="protein1", right_on="protein", how="inner")
string_df = string_df.rename(columns={"uniprot": "protein1_uniprot"})
string_df = string_df.drop(columns=["protein"])

# Merge for protein2
string_df = string_df.merge(mapping_df, left_on="protein2", right_on="protein", how="inner")
string_df = string_df.rename(columns={"uniprot": "protein2_uniprot"})
string_df = string_df.drop(columns=["protein"])

string_df.head()
mapping_df.head()
final_positive = string_df[["protein1_uniprot", "protein2_uniprot"]].copy()
final_positive["label"] = 1

print("Clean positive interactions:", len(final_positive))



Total high confidence interactions: 473860
Clean positive interactions: 461338


In [5]:
# Randomly sample 5000 positive interactions
positive_sample = final_positive.sample(n=5000, random_state=42).reset_index(drop=True)

print("Sampled positive interactions:", len(positive_sample))
positive_sample.head()
import random

# Get unique proteins
all_proteins = pd.unique(
    positive_sample[["protein1_uniprot", "protein2_uniprot"]].values.ravel()
)

print("Unique proteins in sample:", len(all_proteins))
positive_pairs = set(
    zip(positive_sample["protein1_uniprot"], positive_sample["protein2_uniprot"])
)

negative_pairs = set()

while len(negative_pairs) < 5000:
    p1, p2 = random.sample(list(all_proteins), 2)
    
    # Avoid self-interaction & avoid positive duplicates
    if (p1, p2) not in positive_pairs and (p2, p1) not in positive_pairs:
        negative_pairs.add((p1, p2))

negative_sample = pd.DataFrame(list(negative_pairs), 
                               columns=["protein1_uniprot", "protein2_uniprot"])

negative_sample["label"] = 0

print("Generated negative samples:", len(negative_sample))
dataset = pd.concat([positive_sample, negative_sample], ignore_index=True)

print("Final dataset size:", len(dataset))
dataset.head()


Sampled positive interactions: 5000
Unique proteins in sample: 5313
Generated negative samples: 5000
Final dataset size: 10000


,protein1_uniprot,protein2_uniprot,label
0,Q9NQ66,P49441,1
1,A0A1W2PRB8,A0A087WYL7,1
2,Q71DI3,Q6VMQ6,1
3,O75691,O43818,1
4,Q9NP97,Q8WW35,1


In [6]:
from Bio import SeqIO

# Load FASTA
fasta_file = "uniprotkb_organism_id_9606_AND_reviewed_2026_02_16.fasta"

sequence_dict = {}

for record in SeqIO.parse(fasta_file, "fasta"):
    # Extract UniProt ID (middle part between |)
    uniprot_id = record.id.split("|")[1]
    sequence_dict[uniprot_id] = str(record.seq)

print("Total sequences loaded:", len(sequence_dict))


Total sequences loaded: 20431


In [7]:
dataset["seq1"] = dataset["protein1_uniprot"].map(sequence_dict)
dataset["seq2"] = dataset["protein2_uniprot"].map(sequence_dict)

# Remove rows where sequence is missing
dataset = dataset.dropna(subset=["seq1", "seq2"]).reset_index(drop=True)

print("Final dataset after sequence mapping:", len(dataset))
dataset.head()


Final dataset after sequence mapping: 9072


,protein1_uniprot,protein2_uniprot,label,seq1,seq2
0,Q9NQ66,P49441,1,MAGAQPGVHALQLKPVCVSDSLKKGTKFVKWDDDSTIVTPIILRTD...,MSDILRELLCVSEKAANIARACRQQEALFQLLIEEKKEGEKNKKFA...
1,Q71DI3,Q6VMQ6,1,MARTKQTARKSTGGKAPRKQLATKAARKSAPATGGVKKPHRYRPGT...,MDSLEEPQKKVFKARKTMRVSDRQQLEAVYKVKEELLKTDVKLLNG...
2,O75691,O43818,1,MKTKPVSHKTENTYRFLTFAERLGNVNIDIIHRIDRTASYEEEVET...,MSATAAARKRGKPASGAGAGAGAGKRRRKADSAGDRGKSKGGGKMN...
3,Q9NP97,Q8WW35,1,MAEVEETLKRLQSQKGVQGIIVVNTEGIPIKSTMDNPTTTQYASLM...,MATSIGVSFSVGDGVPEAEKNAGEPENTYILRPVFQQRFRPSVVKD...
4,P07948,P43403,1,MGCIKSKGKDSLSDDGVDLKTQPVRNTERTIYVRDPTSNKQQRPVP...,MPDPAAHLPFFYGSISRAEAEEHLKLAGMADGLFLLRQCLRSLGGY...


In [8]:
import torch
print("CUDA available:", torch.cuda.is_available())


CUDA available: False


In [9]:
from transformers import AutoTokenizer, AutoModel
import torch

model_name = "facebook/esm2_t6_8M_UR50D"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

model.eval()  # evaluation mode

print("Model loaded successfully")


Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded successfully


In [10]:
test_seq = dataset["seq1"].iloc[0]
embedding = get_embedding(test_seq)

print("Embedding shape:", embedding.shape)


NameError: name 'get_embedding' is not defined